# ЛР3. Прокси XML-RPC и ограничение запросов

Требуется передавать запросы клиента основному серверу и регистрировать время
вызова, название операции и длительность её выполнения. Прокси работает на порту
8009, основной сервер — 8008, статистика SQLite — 8018.

По умолчанию допускается 5 запросов за 5 секунд. Следующий запрос ожидает
освобождения окна и выполняется автоматически. Для него сохраняются событие
`too_many_requests` и затем результат обычной операции. Потерянные при выключенной
статистике события повторно не отправляются.

Запустите после основного сервера и перед клиентом. Перед повторным запуском
перезапустите ядро. `system.listMethods` обслуживается самим прокси без ограничения.


In [1]:
from datetime import datetime
from http.client import HTTPConnection
from xmlrpc.server import SimpleXMLRPCRequestHandler, SimpleXMLRPCServer
import time
import xmlrpc.client


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)


maximum_requests = 5
time_window_seconds = 5
statistics_timeout_seconds = 0.2
request_times = []
main_service = xmlrpc.client.ServerProxy('http://127.0.0.1:8008/RPC2', allow_none=True)

if maximum_requests < 1 or time_window_seconds <= 0:
    raise ValueError('Количество запросов и длина окна должны быть положительными')


# Необязательная статистика: короткий тайм-аут, ошибка не прерывает операцию.
def send_statistics_event(event_type, timestamp, duration, details=''):
    statistics_connection = HTTPConnection('127.0.0.1', 8018, timeout=statistics_timeout_seconds)
    try:
        request_body = xmlrpc.client.dumps(
            (event_type, timestamp, duration, details), methodname='add_event', allow_none=True
        )
        statistics_connection.request('POST', '/RPC2', request_body, {'Content-Type': 'text/xml'})
        response = statistics_connection.getresponse()
        response_body = response.read()
        if response.status != 200:
            raise OSError('Ошибка HTTP сервиса статистики')
        xmlrpc.client.loads(response_body)
    except Exception as error:
        print('Статистика недоступна, событие пропущено:', event_type, str(error), flush=True)
    finally:
        statistics_connection.close()


# Общее скользящее окно для всех прикладных методов и клиентов прокси.
# Сервер однопоточный: ожидающий запрос выполняется перед следующим.
def wait_for_request_slot(method_name):
    limit_event_sent = False
    while True:
        current_time = time.monotonic()
        request_times[:] = [saved_time for saved_time in request_times
                            if current_time - saved_time < time_window_seconds]
        if len(request_times) < maximum_requests:
            request_times.append(current_time)
            return
        if not limit_event_sent:
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            send_statistics_event('too_many_requests', timestamp, 0.0, 'method=' + method_name)
            limit_event_sent = True
        # Повторно вычисляем остаток после отправки события статистики.
        wait_seconds = request_times[0] + time_window_seconds - time.monotonic()
        if wait_seconds > 0:
            time.sleep(wait_seconds)


# Время ожидания лимита и журналирования не входит в длительность вызова.
def call_main_service(method_name, *arguments):
    wait_for_request_slot(method_name)
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    started_at = time.perf_counter()
    try:
        result = getattr(main_service, method_name)(*arguments)
    except Exception as error:
        duration = time.perf_counter() - started_at
        send_statistics_event(method_name, timestamp, duration, 'error: ' + str(error))
        if isinstance(error, xmlrpc.client.Fault):
            raise
        raise xmlrpc.client.Fault(503, 'Основной сервер недоступен: ' + str(error))
    duration = time.perf_counter() - started_at
    send_statistics_event(method_name, timestamp, duration)
    return result


def ping():
    return call_main_service('ping')


def now():
    return call_main_service('now')


def show_type(argument):
    return call_main_service('type', argument)


def test_sum(first_number, second_number):
    return call_main_service('sum', first_number, second_number)


def test_pow(base_number, exponent):
    return call_main_service('pow', base_number, exponent)


def black_list_check(surname, name, patronym, birth):
    return call_main_service('black_list_check', surname, name, patronym, birth)


def color_inversion(binary_image):
    return call_main_service('color_inversion', binary_image)


def binarize_image(binary_image, threshold):
    return call_main_service('binarize_image', binary_image, threshold)


def binarize_with_coverage(binary_image, threshold):
    return call_main_service('binarize_with_coverage', binary_image, threshold)


def vertical_flip(binary_image):
    return call_main_service('vertical_flip', binary_image)


proxy_server = SimpleXMLRPCServer(
    ('127.0.0.1', 8009), requestHandler=RequestHandler, allow_none=True, logRequests=True
)
proxy_server.register_introspection_functions()
proxy_server.register_function(ping, 'ping')
proxy_server.register_function(now, 'now')
proxy_server.register_function(show_type, 'type')
proxy_server.register_function(test_sum, 'sum')
proxy_server.register_function(test_pow, 'pow')
proxy_server.register_function(black_list_check, 'black_list_check')
proxy_server.register_function(color_inversion, 'color_inversion')
proxy_server.register_function(binarize_image, 'binarize_image')
proxy_server.register_function(binarize_with_coverage, 'binarize_with_coverage')
proxy_server.register_function(vertical_flip, 'vertical_flip')

print('Прокси XML-RPC слушает порт 8009', flush=True)
print('Ограничение:', maximum_requests, 'запросов за', time_window_seconds, 'секунд', flush=True)
try:
    proxy_server.serve_forever()
except KeyboardInterrupt:
    print("Сервер остановлен")
finally:
    proxy_server.server_close()


Прокси XML-RPC слушает порт 8009
Ограничение: 5 запросов за 5 секунд
Прокси XML-RPC слушает порт 8009
Ограничение: 5 запросов за 5 секунд
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:52] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:38:57] "POST /RPC2 HTTP/1.1" 200 -
Прокси XML-RPC слушает порт 8009
Ограничение: 5 запросов за 5 секунд
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [20/Oct/2026 17:39:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - -